# Whole-image registration with `ImageRegistration`

`ImageRegistration` estimates an affine transform, then a nonlinear warp, and atomically commits each reusable stage. Create a new instance for each moving round.

In [ ]:
# EDIT THESE PATHS and the round number before running the notebook.
from pathlib import Path

FIXED_IMAGE = Path("path/to/fixed_reference.tif")
MOVING_IMAGE = Path("path/to/moving_round.tif")
OUTPUT_DIRECTORY = Path("path/to/registration_output")
ROUND_NUMBER = 2
PIXEL_SIZE_XYZ = (0.15, 0.15, 0.15)
ITK_THREADS = 1
BLAS_THREADS = 1

Spacing is in micrometers and XYZ order. The conservative one-thread settings reflect the stable default; increase them only after validating the target environment and dataset.

In [ ]:
from cyexm.registration import ImageRegistration
from cyexm.runtime import NativeThreadConfig

registration = ImageRegistration(
    fixed_image_path=FIXED_IMAGE,
    moving_image_path=MOVING_IMAGE,
    save_directory=OUTPUT_DIRECTORY,
    imaging_round=ROUND_NUMBER,
    pixel_size=PIXEL_SIZE_XYZ,
    native_threads=NativeThreadConfig(ITK_THREADS, BLAS_THREADS),
    linear_accuracy="low",
    nonlinear_accuracy="high",
)

In [ ]:
registration.register()

Successful registration creates `linear_transform_{round}.mat`, `nonlinear_warp_{round}.nii.gz`, and `nonlinear_registered_{round}.tif`, plus metrics and completion records. Rerunning the same request validates and reuses those records.

In [ ]:
artifacts = {
    "affine": OUTPUT_DIRECTORY / f"linear_transform_{ROUND_NUMBER}.mat",
    "warp": OUTPUT_DIRECTORY / f"nonlinear_warp_{ROUND_NUMBER}.nii.gz",
    "registered": OUTPUT_DIRECTORY / f"nonlinear_registered_{ROUND_NUMBER}.tif",
}
artifacts